# 03 · Modelling table

Turns the daily series from notebook 02 into the table used for training and evaluation. One row is one district-day, described only by conditions observed up to and including that day, and labelled by whether a flood began in the following three days.

**Input:** the outputs of notebooks 01 and 02.

**Output:** `model_table.parquet`, `split_counts.csv`, `feature_contrast.csv`, `fig_feature_contrast.png`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

WORK = Path("/kaggle/working")
OUT = WORK / "outputs"
OUT.mkdir(exist_ok=True)

HORIZON = 3
API_DECAY = 0.9
WINDOWS = (1, 3, 7, 14, 30)
SEASON_MONTHS = (6, 10)
SPLITS = {"train": (2004, 2016), "val": (2017, 2019), "test": (2020, 2023)}

FEATURES = [
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "rain_14d",
    "rain_30d",
    "rain_api",
    "soil_moisture",
    "soil_change_7d",
    "discharge",
    "discharge_change_3d",
    "discharge_anomaly",
]

In [ ]:
weather_dir = next(p.parent for p in Path("/kaggle/input").rglob("up_daily_features.parquet"))
label_dir = next(p.parent for p in Path("/kaggle/input").rglob("up_flood_labels.csv"))

daily = pd.read_parquet(weather_dir / "up_daily_features.parquet")
labels = pd.read_csv(label_dir / "up_flood_labels.csv", parse_dates=["start", "end"])
labels = labels[labels["district"].isin(daily["district"].unique())]

daily = daily.sort_values(["district", "date"]).reset_index(drop=True)
print(f"{len(daily)} district-days, {daily['district'].nunique()} districts")
print(f"{len(labels)} flood records in these districts")

## Features

Each feature summarises the catchment state up to the end of day *t*: accumulated rainfall over five windows, an antecedent precipitation index that decays older rain geometrically, surface soil moisture and its weekly change, and river discharge with its three-day change.

Rolling windows are computed within a district and a single monsoon season. The downloaded series jumps from 31 October to 1 May, so windows that spanned that gap would mix one year's October with the next year's May. Grouping by season also means the 30-day window is undefined until 30 May, which is why May is downloaded but excluded below.

In [ ]:
def build_features(frame):
    frame = frame.sort_values(["district", "date"]).copy()
    season = frame.groupby([frame["district"], frame["date"].dt.year], sort=False)

    for window in WINDOWS:
        frame[f"rain_{window}d"] = season["precip_mm"].transform(lambda s, w=window: s.rolling(w, min_periods=w).sum())
    frame["rain_api"] = season["precip_mm"].transform(lambda s: s.ewm(alpha=1 - API_DECAY, adjust=False).mean())
    frame["soil_change_7d"] = season["soil_moisture"].transform(lambda s: s - s.shift(7))
    frame["discharge_change_3d"] = season["discharge"].transform(lambda s: s - s.shift(3))
    return frame


table = build_features(daily)
table[["date", "precip_mm", "rain_7d", "rain_api", "discharge_change_3d"]].tail(3)

## Target and exclusions

The prediction is issued at the end of day *t* and asks whether a flood begins on any of days *t+1* to *t+3*. An onset on day *t* itself is not part of the target, so no feature and its label can describe the same day.

Three groups of rows are removed. Days on which a flood is already under way are neither onsets nor quiet days, and keeping them as negatives would teach the model that flood conditions imply no flood. May is removed once it has served as history for June. The last three days of each season are removed because their outcome falls outside the downloaded window and cannot be known.

`oracle_rain_next3d` deliberately looks ahead and is excluded from `FEATURES`. It exists only for the perfect-forecast comparison in notebook 04.

In [ ]:
onsets = labels[["district", "start"]].rename(columns={"start": "date"}).drop_duplicates()
onsets["onset"] = 1
table = table.merge(onsets, on=["district", "date"], how="left")
table["onset"] = table["onset"].fillna(0)

season = table.groupby([table["district"], table["date"].dt.year], sort=False)
future_onset = sum(season["onset"].shift(-step) for step in range(1, HORIZON + 1))
future_rain = sum(season["precip_mm"].shift(-step) for step in range(1, HORIZON + 1))

table["target"] = (future_onset > 0).astype("int8")
table["oracle_rain_next3d"] = future_rain
table = table[future_onset.notna()]

ongoing = pd.concat(
    pd.DataFrame({"district": row.district, "date": pd.date_range(row.start, row.end)}) for row in labels.itertuples()
).drop_duplicates()
ongoing["ongoing"] = True

table = table.merge(ongoing, on=["district", "date"], how="left")

rows = [("District-days downloaded", len(daily))]
table = table[table["date"].dt.month.between(*SEASON_MONTHS)]
rows.append(("June to October", len(table)))
table = table[table["ongoing"].isna()]
rows.append(("Excluding days already flooding", len(table)))
table = table.dropna(subset=[f for f in FEATURES if f != "discharge_anomaly"])
rows.append(("Complete features", len(table)))

table = table.drop(columns=["ongoing", "onset"]).reset_index(drop=True)
pd.DataFrame(rows, columns=["step", "rows"])

## Splits and class balance

One onset marks up to three rows as positive, so positives outnumber flood records. Recorded flood density rises over the study period, so the positive rate differs between splits; notebook 04 reports skill against a climatology baseline fitted per split for that reason.

Years are split chronologically, so every evaluation predicts seasons the model has not seen. A random split would place neighbouring days of the same flood on both sides of the boundary and inflate the results.

In [ ]:
def assign_split(year):
    for name, (first, last) in SPLITS.items():
        if first <= year <= last:
            return name
    return None


table["split"] = table["date"].dt.year.map(assign_split)
table = table[table["split"].notna()].reset_index(drop=True)
table["split"].value_counts().reindex(SPLITS)

## Discharge anomaly

Absolute discharge means different things in different districts: 4,000 m³/s is ordinary on the Ganga and extreme on a tributary. The anomaly divides discharge by the median for that district and calendar month, computed from the training years alone so that no information from the validation or test seasons reaches the training features.

In [ ]:
table["month"] = table["date"].dt.month
typical = table[table["split"] == "train"].groupby(["district", "month"])["discharge"].median().rename("typical")

table = table.join(typical, on=["district", "month"])
table["discharge_anomaly"] = table["discharge"] / table["typical"]
missing = table["discharge_anomaly"].isna().sum()
table = table.dropna(subset=["discharge_anomaly"]).drop(columns=["month", "typical"]).reset_index(drop=True)

print(f"{missing} rows dropped for having no training climatology")
table["discharge_anomaly"].describe().round(2)

In [ ]:
counts = table.groupby("split").agg(rows=("target", "size"), positives=("target", "sum"))
counts["positive_rate"] = (counts["positives"] / counts["rows"]).round(5)
counts = counts.reindex(SPLITS)
counts

## Leakage test

Every rolling feature is recomputed for a sample of rows from a history truncated at that row's own date. If any value differs from the one in the table, a feature is reading forward in time and the notebook stops.

In [ ]:
rolling_features = [f for f in FEATURES if f not in ("soil_moisture", "discharge", "discharge_anomaly")]
sample = table.sample(8, random_state=0)

for row in sample.itertuples():
    past = daily[(daily["district"] == row.district) & (daily["date"] <= row.date)]
    recomputed = build_features(past).iloc[-1]
    for name in rolling_features:
        if not np.isclose(recomputed[name], getattr(row, name)):
            raise AssertionError(f"{name} changes when future data is removed ({row.district} {row.date.date()})")

print(f"{len(sample)} rows recomputed from truncated history, {len(rolling_features)} features each, all identical")

## Feature contrast

Mean feature values on days followed by a flood against days that are not. Antecedent rainfall and discharge anomaly are expected to be markedly higher before floods; if they are not, the features carry no signal and no model can recover it.

In [ ]:
contrast = table.groupby("target")[FEATURES].mean().T
contrast.columns = ["no flood", "flood within 3 days"]
contrast["ratio"] = (contrast["flood within 3 days"] / contrast["no flood"]).round(2)
contrast.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, column, title in zip(
    axes, ["rain_7d", "discharge_anomaly"], ["7-day rainfall (mm)", "Discharge vs district normal"]
):
    data = [table.loc[table["target"] == value, column].dropna() for value in (0, 1)]
    ax.boxplot(data, showfliers=False)
    ax.set_xticklabels(["no flood", "flood"])
    ax.set_title(title)
fig.tight_layout()
fig.savefig(OUT / "fig_feature_contrast.png", dpi=200)

In [ ]:
table.to_parquet(OUT / "model_table.parquet", index=False)
contrast.to_csv(OUT / "feature_contrast.csv")
counts.to_csv(OUT / "split_counts.csv")
sorted(p.name for p in OUT.iterdir())